#  College Chatbot using NLP

An intent-classification based college enquiry chatbot covering **B.Tech, BCA, B.Sc, M.Tech and MCA**
programs, wrapped in an interactive **Gradio dashboard**.

**Pipeline:** Text Cleaning → Lemmatization → TF-IDF Vectorization → Cosine Similarity Intent
Matching (+ Logistic Regression classifier for evaluation) → Response Retrieval

**Dashboard tabs:**
1.  **Chatbot** — talk to the bot
2.  **Analytics Dashboard** — live charts on chat usage
3.  **Dataset Explorer** — browse/search the training dataset
4. **Model Insights** — accuracy, confusion matrix



In [1]:
import re
import string
import random
from datetime import datetime

import nltk
import pandas as pd
import numpy as np
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


In [2]:

for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4", "stopwords"]:
    try:
        nltk.data.find(pkg)
    except LookupError:
        nltk.download(pkg, quiet=True)

from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()
STOPWORDS = set(stopwords.words("english")) - {"what", "how", "when", "where", "who", "is", "are"}

DATASET_PATH = "college_chatbot_dataset.csv"
CONFIDENCE_THRESHOLD = 0.20


## 1. Text Preprocessing

Lowercase → strip punctuation/numbers → remove stopwords → lemmatize each token using NLTK's
WordNet lemmatizer. This cleaned text is what actually gets vectorized.


In [3]:
def clean_text(text: str) -> str:
    """Lowercase, strip punctuation/numbers, remove stopwords, lemmatize."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)


clean_text("What is the Fee Structure for B.Tech admission?")


'what is fee structure tech admission'

## 2. Load Data & Build the Model

`CollegeChatbot` loads the CSV dataset, builds a TF-IDF vector space over all training patterns,
and fits two models:
- A **cosine-similarity retriever** (used live, at inference time) — matches a new query to the
  closest known training pattern.
- A **Logistic Regression classifier** trained on a held-out split — used purely for the
  Model Insights tab (accuracy, confusion matrix).


In [5]:
class CollegeChatbot:
    def __init__(self, csv_path: str):
        self.df = pd.read_csv(csv_path)
        self.df["clean_pattern"] = self.df["pattern"].apply(clean_text)


        self.tag_to_response = (
            self.df.drop_duplicates("tag").set_index("tag")["response"].to_dict()
        )
        self.tag_to_category = (
            self.df.drop_duplicates("tag").set_index("tag")["category"].to_dict()
        )


        self.vectorizer = TfidfVectorizer(ngram_range=(1, 1), min_df=1)
        self.X = self.vectorizer.fit_transform(self.df["clean_pattern"])


        self.clf = LogisticRegression(max_iter=1000)
        self.clf.fit(self.X, self.df["tag"])


        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.df["tag"], test_size=0.25, random_state=42, stratify=self.df["tag"]
        )
        eval_clf = LogisticRegression(max_iter=1000)
        eval_clf.fit(X_train, y_train)
        self.y_test = y_test
        self.y_pred = eval_clf.predict(X_test)
        self.eval_accuracy = accuracy_score(y_test, self.y_pred)


        self.chat_log = []


    def predict(self, user_text: str):
        """Return (tag, response, confidence, category) for a user message."""
        cleaned = clean_text(user_text)
        if cleaned.strip() == "":
            return None, "Could you please rephrase your question?", 0.0, "Unknown"

        vec = self.vectorizer.transform([cleaned])
        sims = cosine_similarity(vec, self.X).flatten()
        best_idx = sims.argmax()
        confidence = float(sims[best_idx])
        tag = self.df.iloc[best_idx]["tag"]

        if confidence < CONFIDENCE_THRESHOLD:
            response = (
                "I'm sorry, I couldn't find an exact answer to that. "
                "Please try asking about admissions, courses, fees, scholarships, "
                "departments, academic calendar, exams, placements or hostel."
            )
            category = "Unknown"
            tag = "fallback"
        else:
            response = self.tag_to_response[tag]
            category = self.tag_to_category[tag]

        self.chat_log.append({
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "query": user_text,
            "tag": tag,
            "category": category,
            "confidence": round(confidence, 3),
        })
        return tag, response, confidence, category


In [6]:
bot = CollegeChatbot('/content/college_chatbot_dataset (2).csv')
print(f"Loaded {len(bot.df)} training patterns across {bot.df['tag'].nunique()} intents.")
print(f"Hold-out classifier accuracy: {bot.eval_accuracy*100:.2f}%")


Loaded 155 training patterns across 38 intents.
Hold-out classifier accuracy: 30.77%


### Quick test — try a few sample queries

In [7]:
tests = [
    "What is the fee for MCA?",
    "Tell me about BTech admission",
    "Which companies come for placements?",
    "Is hostel available for girls?",
    "hello",
    "random gibberish xyz",
]
for t in tests:
    tag, resp, conf, cat = bot.predict(t)
    print(f"Q: {t}\n -> tag={tag} | confidence={conf:.2f} | category={cat}\n -> {resp}\n")


Q: What is the fee for MCA?
 -> tag=fee_mca | confidence=1.00 | category=Fee Structure
 -> MCA annual tuition fee is approximately Rs. 70,000 per year, excluding hostel and examination charges.

Q: Tell me about BTech admission
 -> tag=admission_btech | confidence=0.64 | category=Admission Information
 -> B.Tech admission requires 10+2 with Physics, Chemistry and Mathematics, and a valid entrance exam score (JEE/State CET). Apply online through the admissions portal.

Q: Which companies come for placements?
 -> tag=placement_companies | confidence=0.82 | category=Placement Information
 -> Recruiters include TCS, Infosys, Wipro, Cognizant, Accenture and several product-based startups, visiting every year during the placement season.

Q: Is hostel available for girls?
 -> tag=hostel_availability | confidence=0.75 | category=Hostel Information
 -> Separate hostel facilities are available for boys and girls within the campus, with limited seats allotted on a first-come-first-served basis.


## 3. Chat Function for Gradio

In [8]:
def chat_fn(message, history):
    _, response, confidence, category = bot.predict(message)
    tag_note = f"\n\n_(intent: {category} | confidence: {confidence:.2f})_"
    return response + tag_note


def clear_log():
    bot.chat_log = []
    return "Chat log cleared."


## 4. Analytics Dashboard Functions\n\nThese build the live Plotly charts shown in the **Analytics Dashboard** tab, based on the in-memory chat log.

In [9]:
def build_category_chart():
    if not bot.chat_log:
        return px.bar(title="No conversations yet - chat with the bot first")
    log_df = pd.DataFrame(bot.chat_log)
    counts = log_df["category"].value_counts().reset_index()
    counts.columns = ["Category", "Count"]
    fig = px.bar(counts, x="Category", y="Count", title="Questions Asked by Category",
                 color="Category")
    fig.update_layout(showlegend=False)
    return fig


def build_confidence_chart():
    if not bot.chat_log:
        return px.line(title="No conversations yet - chat with the bot first")
    log_df = pd.DataFrame(bot.chat_log)
    fig = px.line(log_df, y="confidence", markers=True,
                   title="Model Confidence Over Recent Queries")
    fig.add_hline(y=CONFIDENCE_THRESHOLD, line_dash="dash",
                  annotation_text="Fallback threshold", line_color="red")
    return fig


def build_log_table():
    if not bot.chat_log:
        return pd.DataFrame(columns=["timestamp", "query", "tag", "category", "confidence"])
    return pd.DataFrame(bot.chat_log).iloc[::-1]  # most recent first


## 5. Dataset Explorer Functions\n\nPower the **Dataset Explorer** tab: category/course-level charts plus keyword search.

In [10]:
def dataset_overview_chart():
    counts = bot.df["category"].value_counts().reset_index()
    counts.columns = ["Category", "Number of Training Patterns"]
    fig = px.bar(counts, x="Number of Training Patterns", y="Category", orientation="h",
                 title="Training Dataset - Patterns per Feature Category", color="Category")
    fig.update_layout(showlegend=False)
    return fig


def course_level_chart():
    sub = bot.df[bot.df["course_level"] != "General"]
    counts = sub["course_level"].value_counts().reset_index()
    counts.columns = ["Course Level", "Patterns"]
    fig = px.pie(counts, names="Course Level", values="Patterns",
                 title="Training Data Split by Course Level (BTech/BCA/BSc/MTech/MCA)")
    return fig


def search_dataset(keyword, category_filter, course_filter):
    df = bot.df.copy()
    if category_filter and category_filter != "All":
        df = df[df["category"] == category_filter]
    if course_filter and course_filter != "All":
        df = df[df["course_level"] == course_filter]
    if keyword:
        kw = keyword.lower()
        df = df[df["pattern"].str.lower().str.contains(kw) |
                df["response"].str.lower().str.contains(kw)]
    return df[["tag", "category", "course_level", "pattern", "response"]]


## 6. Model Insights Functions\n\nPower the **Model Insights** tab: accuracy summary and confusion matrix heatmap.

In [11]:
def model_accuracy_text():
    return (
        f"### Model Evaluation\n"
        f"- Algorithm: TF-IDF + Logistic Regression / Cosine Similarity\n"
        f"- Train/Test split: 75% / 25% (stratified)\n"
        f"- **Hold-out accuracy: {bot.eval_accuracy*100:.2f}%**\n"
        f"- Total intents (tags): {bot.df['tag'].nunique()}\n"
        f"- Total training patterns: {len(bot.df)}\n"
    )


def confusion_matrix_chart():
    labels = sorted(bot.df["tag"].unique())
    cm = confusion_matrix(bot.y_test, bot.y_pred, labels=labels)
    fig = go.Figure(data=go.Heatmap(z=cm, x=labels, y=labels, colorscale="Blues"))
    fig.update_layout(title="Confusion Matrix (Hold-out Test Set)",
                       xaxis_title="Predicted Tag", yaxis_title="Actual Tag",
                       height=700)
    return fig


## 7. Build the Gradio Dashboard\n\nA single `gr.Blocks` app with four tabs. Run the next cell, then run `demo.launch()` in the cell after to start the dashboard (it will render inline in the notebook, or open in your browser).

In [13]:
CATEGORY_CHOICES = ["All"] + sorted(bot.df["category"].unique().tolist())
COURSE_CHOICES = ["All"] + sorted(bot.df["course_level"].unique().tolist())

with gr.Blocks(title="College Chatbot Dashboard") as demo:
    gr.Markdown(
        """
        #  College Chatbot - NLP Dashboard
        An intent-classification chatbot for B.Tech, BCA, B.Sc, M.Tech and MCA enquiries,
        covering College Info, Courses, Admissions, Fees, Scholarships, Departments,
        Academic Calendar, Exams, Placements and Hostel information.
        """
    )

    with gr.Tabs():

        with gr.Tab(" Chatbot"):
            gr.ChatInterface(
                fn=chat_fn,
                examples=[
                    "What is the fee for MCA?",
                    "Tell me about BTech admission",
                    "Which companies come for placements?",
                    "Is hostel available for girls?",
                    "What courses are offered in BSc?",
                ],
            )


        with gr.Tab(" Analytics Dashboard"):
            gr.Markdown("Live analytics based on the conversation happening in the Chatbot tab.")
            refresh_btn = gr.Button(" Refresh Analytics")
            with gr.Row():
                cat_plot = gr.Plot(label="Category-wise Questions")
                conf_plot = gr.Plot(label="Confidence Trend")
            log_table = gr.Dataframe(label="Recent Chat Log", interactive=False)
            clear_btn = gr.Button(" Clear Chat Log")
            clear_status = gr.Textbox(label="Status", interactive=False)

            refresh_btn.click(fn=build_category_chart, outputs=cat_plot)
            refresh_btn.click(fn=build_confidence_chart, outputs=conf_plot)
            refresh_btn.click(fn=build_log_table, outputs=log_table)
            clear_btn.click(fn=clear_log, outputs=clear_status)


        with gr.Tab(" Dataset Explorer"):
            gr.Markdown("Explore the labelled training dataset used to train the intent classifier.")
            with gr.Row():
                overview_plot = gr.Plot(value=dataset_overview_chart())
                course_plot = gr.Plot(value=course_level_chart())

            gr.Markdown("### Search the dataset")
            with gr.Row():
                keyword_box = gr.Textbox(label="Keyword search")
                category_drop = gr.Dropdown(CATEGORY_CHOICES, value="All", label="Category")
                course_drop = gr.Dropdown(COURSE_CHOICES, value="All", label="Course Level")
            search_btn = gr.Button("Search")
            result_table = gr.Dataframe(value=bot.df[["tag", "category", "course_level",
                                                        "pattern", "response"]],
                                         label="Dataset", interactive=False)
            search_btn.click(fn=search_dataset,
                              inputs=[keyword_box, category_drop, course_drop],
                              outputs=result_table)


        with gr.Tab(" Model Insights"):
            gr.Markdown(model_accuracy_text())
            gr.Plot(value=confusion_matrix_chart())

print("Gradio app built. Run the next cell to launch it.")


Gradio app built. Run the next cell to launch it.


## 8. Launch the Dashboard\n\nThis opens the dashboard inline in the notebook (and prints a local URL you can also open in your browser). Use `demo.launch(share=True)` instead if you want a public shareable link.

In [14]:
demo.launch(theme=gr.themes.Soft())


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d61aa250c01c078d7e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
